In [1]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
data = pd.read_csv('./dataset/credit-a.csv', header=None)

In [3]:
data.head()

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15
0,0,30.83,0.000,0,0,9,0,1.25,0,0,1,1,0,202,0.0,-1
1,1,58.67,4.460,0,0,8,1,3.04,0,0,6,1,0,43,560.0,-1
2,1,24.50,0.500,0,0,8,1,1.50,0,1,0,1,0,280,824.0,-1
3,0,27.83,1.540,0,0,9,0,3.75,0,0,5,0,0,100,3.0,-1
4,0,20.17,5.625,0,0,9,0,1.71,0,1,0,1,2,120,0.0,-1


In [4]:
# 前15列是特征, 最后一列是标记
X = data.iloc[:, :-1]
X.shape

(653, 15)

In [5]:
Y = data.iloc[:, -1]

In [7]:
# series 是不能作为标记的
Y

0     -1
1     -1
2     -1
3     -1
4     -1
      ..
648    1
649    1
650    1
651    1
652    1
Name: 15, Length: 653, dtype: int64

In [8]:
# 把标记变成0和1, 方便最后求概率
Y.replace(-1, 0, inplace=True)

In [9]:
Y.value_counts()

15
1    357
0    296
Name: count, dtype: int64

In [10]:
X = torch.from_numpy(X.values).type(torch.FloatTensor)
Y = torch.from_numpy(Y.values.reshape(-1, 1)).type(torch.FloatTensor)

In [11]:
# 回归和分类之间, 区别其实不大. 回归后面加上一层sigmoid, 就变成分类了.
from torch import nn

In [12]:
model = nn.Sequential(
    nn.Linear(15, 1),
    nn.Sigmoid()
)

In [13]:
model

Sequential(
  (0): Linear(in_features=15, out_features=1, bias=True)
  (1): Sigmoid()
)

In [16]:
# BCE binary cross entroy 二分类的交叉熵损失
loss_fn = nn.BCELoss()

In [17]:
opt = torch.optim.SGD(model.parameters(), lr=0.0001)

In [18]:
X.shape

torch.Size([653, 15])

In [19]:
batch_size = 32
step = 653 // 32

In [21]:
for epoch in range(1000):
    # 每次取32个数据
    for batch in range(step):
        # 起始索引
        start = batch * batch_size
        # 结束索引
        end = start + batch_size
        # 取数据
        x = X[start: end]
        y = Y[start: end]
        y_pred = model(x)
        loss = loss_fn(y_pred, y)
        # 梯度清零
        opt.zero_grad()
        # 反向传播
        loss.backward()
        # 更新
        opt.step()

In [22]:
model.state_dict()

OrderedDict([('0.weight',
              tensor([[ 0.1144,  0.0190, -0.0142, -0.1241,  0.2177,  0.0538,  0.0602, -0.1662,
                        0.3504, -0.1602, -0.3551,  0.1352,  0.0380,  0.0104,  0.0071]])),
             ('0.bias', tensor([-0.1156]))])

In [26]:
# 计算正确率
# 设定阈值
# 现在预测得到的是概率, 我们根据阈值, 把概率转化为类型, 就可以计算准确率
((model(X).data.numpy() > 0.5) == Y.numpy()).mean()

np.float64(0.6064318529862175)